# Merge S2 — Train 2 specialist models (cho pseudo-label)

Mục tiêu: có 2 model để điền nhãn thiếu (cross-inference cùng domain OAI):
- **Model_ZIB** (Dataset001, nhãn 0–5) → sau này suy ra **bone (1,3)** trên iMorphics.
- **Model_iMorph** (Dataset013, remap 1–6) → suy ra **meniscus+patella** trên OAI-ZIB.

Đây là specialist cho pseudo-label → **fold 0 + trainer ngắn là đủ** (không cần SOTA). Model 8-class cuối (S5) mới train full ResEnc + 1000ep để vắt sụn.

**Prereq:** đã chạy S1 (split iMorphics → Dataset012/imagesTr là train subset).

Cấu hình I/O (tối ưu Colab):
- `nnUNet_raw` = **local** (Dataset001 symlink từ Drive; Dataset013 build local).
- `nnUNet_preprocessed` = **local** (nhanh, tái tạo được).
- `nnUNet_results` = **Drive** (checkpoint sống sót khi session đứt).


In [ ]:
!pip install -q nnunetv2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0) Env vars + nnUNet_raw local


In [ ]:
import os
os.makedirs("/content/nnUNet_raw", exist_ok=True)
os.environ["nnUNet_raw"]          = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/content/drive/MyDrive/nnUNet_results"
os.makedirs(os.environ["nnUNet_preprocessed"], exist_ok=True)
os.makedirs(os.environ["nnUNet_results"], exist_ok=True)

# Dataset001 (OAI-ZIB): symlink tu Drive (khong copy 12GB)
link = "/content/nnUNet_raw/Dataset001_KneeOA"
if not os.path.exists(link):
    os.symlink("/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA", link)
print("raw:", os.listdir("/content/nnUNet_raw"))


## 1) Build Dataset013_iMorphSpec (remap nhãn → liên tiếp 1–6)

Copy ảnh iMorphics train (Drive→local) + remap `{2:1,4:2,5:3,6:4,7:5,8:6}`. Lưu map ngược để S3 ghép pseudo-label.


In [ ]:
import shutil, json, numpy as np, nibabel as nib
from pathlib import Path

SRC = Path("/content/drive/MyDrive/nnUNet_raw/Dataset012_iMorphics")
D13 = Path("/content/nnUNet_raw/Dataset013_iMorphSpec")
(D13/"imagesTr").mkdir(parents=True, exist_ok=True)
(D13/"labelsTr").mkdir(parents=True, exist_ok=True)

REMAP     = {2:1, 4:2, 5:3, 6:4, 7:5, 8:6}          # union -> specialist
REMAP_INV = {v:k for k,v in REMAP.items()}          # specialist -> union (dung o S3)

cases = sorted(p.name.replace(".nii.gz","") for p in (SRC/"labelsTr").glob("*.nii.gz"))
for c in cases:
    shutil.copy(str(SRC/"imagesTr"/f"{c}_0000.nii.gz"), str(D13/"imagesTr"/f"{c}_0000.nii.gz"))
    li = nib.load(str(SRC/"labelsTr"/f"{c}.nii.gz")); a = np.asanyarray(li.dataobj).astype(np.uint8)
    out = np.zeros_like(a)
    for s,d in REMAP.items(): out[a==s] = d
    nib.save(nib.Nifti1Image(out, li.affine), str(D13/"labelsTr"/f"{c}.nii.gz"))

labels13 = {"background":0,"femoral_cartilage":1,"medial_tibial_cartilage":2,
            "lateral_tibial_cartilage":3,"medial_meniscus":4,"lateral_meniscus":5,"patellar_cartilage":6}
json.dump({"channel_names":{"0":"MRI"},"labels":labels13,"numTraining":len(cases),
           "file_ending":".nii.gz","name":"iMorphSpec"}, open(D13/"dataset.json","w"), indent=2)
json.dump(REMAP_INV, open("/content/drive/MyDrive/nnUNet_raw/imorph_remap_inv.json","w"))
print("Dataset013 build:", len(cases), "cases | labels 0..6")


## 2) plan_and_preprocess (ResEnc-L) cho cả 2 dataset


In [ ]:
!nnUNetv2_plan_and_preprocess -d 1  -pl nnUNetPlannerResEncL --verify_dataset_integrity
!nnUNetv2_plan_and_preprocess -d 13 -pl nnUNetPlannerResEncL --verify_dataset_integrity


## 3) Train specialist (fold 0)

- Model_ZIB: bone dễ (~0.97) → **250ep** đủ cho pseudo bone.
- Model_iMorph: meniscus khó hơn → **500ep**.
- Nếu Colab đứt: chạy lại **đúng lệnh + thêm `--c`** để resume (checkpoint ở Drive nnUNet_results).


In [ ]:
# Model_ZIB (fold 0) — dung de pseudo-label bone tren iMorphics
!nnUNetv2_train 1 3d_fullres 0 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs


In [ ]:
# Model_iMorph (fold 0) — dung de pseudo-label meniscus+patella tren OAI-ZIB
!nnUNetv2_train 13 3d_fullres 0 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_500epochs


## Ghi chú
- Xong 2 model này → sang **S3**: inference chéo tạo pseudo-label, rồi **S4** ghép thành `Dataset020_KneeUnion` (8-class), **S5** train model cuối (ResEnc + **1000ep** + 5-fold — nơi vắt sụn tối đa) + surface metrics ở **S6**.
- Muốn train nhanh hơn nữa cho specialist: giảm còn `nnUNetTrainer_100epochs`, nhưng meniscus pseudo có thể nhiễu hơn.
